# 🩺 Diabetes HbA1c Prediction using AutoML

**🎯 Objective**: Predict post-treatment HbA1c levels using state-of-the-art machine learning on comprehensive diabetes intervention data.

**📊 Dataset**: 6,990+ participants across 3 intervention studies with 80+ clinical features including:
- Laboratory values (FBS, PPBS, HbA1c, lipid panels)
- Anthropometric measurements (BMI, waist-hip ratio, vital signs) 
- Demographics and lifestyle factors
- Family history and medical backgrounds

**🤖 Method**: H2O AutoML with automated algorithm comparison, hyperparameter optimization, and clinical validation metrics

**🎯 Target Performance**: R² > 0.85, RMSE < 0.5 HbA1c units, Clinical Accuracy > 80% for deployment readiness

**⏱️ Expected Runtime**: 15-20 minutes for complete pipeline execution (2-hour AutoML budget)

## 1. Environment Setup & Package Installation

Configure the runtime before executing the rest of the notebook. Complete the data access step, then install the dependencies so the remaining cells run without interruption.

## 🤖 Auto-sklearn Diabetes HbA1c Prediction

This notebook implements a **sequential training approach** using Auto-sklearn to predict post-treatment HbA1c levels from comprehensive diabetes intervention data.

### 📊 **Project Overview**
- **Objective**: Predict HbA1c levels after diabetes interventions
- **Method**: Auto-sklearn with sequential dataset training
- **Data**: 3 intervention studies with 6,990+ patients
- **Approach**: Train individual models per dataset + combined ensemble

### 🎯 **Key Features**
- **Automated ML**: No manual hyperparameter tuning required
- **Sequential Training**: Each dataset gets specialized model
- **Model Comparison**: Performance evaluation across approaches
- **Clinical Validation**: HbA1c accuracy within ±0.5% and ±1.0%
- **Deployment Ready**: Saved models with performance metrics

### 🚀 **Execution Steps**
1. **Install packages** (Auto-sklearn ecosystem)
2. **Import libraries** and configure settings
3. **Load datasets** sequentially
4. **Train models** (30 minutes per dataset)
5. **Compare performance** and select best model
6. **Generate reports** and save for deployment

**Expected Total Runtime**: 1.5-2 hours for complete analysis

In [ ]:
# =============================================================================
# CORE IMPORTS & GLOBAL SETTINGS
# =============================================================================

import os
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

# Core Auto-sklearn and ML libraries
import autosklearn.regression
import autosklearn.metrics
from autosklearn.experimental.askl2 import AutoSklearn2Regressor

# Scikit-learn ecosystem
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Data manipulation and analysis
import pandas as pd
import numpy as np
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Feature engineering and preprocessing
try:
    from feature_engine.encoding import OrdinalEncoder, OneHotEncoder
    from feature_engine.imputation import MeanMedianImputer
    from feature_engine.selection import DropCorrelatedFeatures
    print("✅ Feature-engine available for advanced preprocessing")
except ImportError:
    print("⚠️  Feature-engine not available - using sklearn preprocessing")

# Interpretability
try:
    import shap
    print("✅ SHAP available for model interpretability")
except ImportError:
    print("⚠️  SHAP not available - install for interpretability analysis")

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Global plotting style for consistency
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (12, 8),
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "font.family": "sans-serif",
})

# Auto-sklearn configuration
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

print("✅ Auto-sklearn ecosystem imported successfully.")
print(f"📈 Scikit-learn version: {sklearn.__version__}")
print(f"🤖 Auto-sklearn ready for automated machine learning")

## 3. Auto-sklearn Core Imports
Import the Auto-sklearn ecosystem and configure the environment for automated machine learning.

In [ ]:
# =============================================================================
# CORE IMPORTS & GLOBAL SETTINGS
# =============================================================================

import os
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import h2o
from h2o.automl import H2OAutoML  # Added missing H2OAutoML import
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Added missing metrics

# Global plotting style for consistency
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("✅ Core libraries imported successfully.")

## 4. Auto-sklearn Configuration
Configure Auto-sklearn parameters, data paths, and sequential training settings.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION FOR AUTO-SKLEARN
# =============================================================================

NOTEBOOK_CONFIG = {
    # Dataset configuration
    "dataset_files": [
        "nmbfinalDiabetes (4)_selected_columns_cleaned_processed_final_imputed.csv",
        "nmbfinalnewDiabetes (3)_selected_columns_cleaned_processed_final_imputed.csv",
        "PrePostFinal (3)_selected_columns_cleaned_processed_final_imputed.csv",
    ],
    "target_column": "PostBLHBA1C",
    
    # Sequential training configuration
    "sequential_training": {
        "train_individual_models": True,
        "combine_datasets": True,
        "model_comparison": True,
        "ensemble_final_models": True,
    },
    
    # Data preprocessing
    "preprocessing": {
        "handle_missing": "median",  # median, mean, mode
        "scale_features": "robust",   # standard, robust, minmax
        "encode_categoricals": "onehot",  # onehot, ordinal, target
        "feature_selection": True,
        "remove_outliers": False,
    },
    
    # Auto-sklearn configuration
    "autosklearn": {
        "time_left_for_this_task": 1800,  # 30 minutes per dataset
        "per_run_time_limit": 300,        # 5 minutes per model
        "memory_limit": 6144,             # 6GB memory limit
        "n_jobs": -1,                     # Use all cores
        "metric": autosklearn.metrics.r2, # R² for regression
        "scoring_functions": [
            autosklearn.metrics.r2,
            autosklearn.metrics.mean_squared_error,
            autosklearn.metrics.mean_absolute_error
        ],
        "resampling_strategy": "cv",      # cross-validation
        "resampling_strategy_arguments": {"folds": 5},
        "ensemble_size": 20,              # Number of models in ensemble
        "ensemble_nbest": 5,              # Top models for ensemble
        "include_estimators": [
            "random_forest",
            "gradient_boosting",
            "extra_trees",
            "adaboost",
            "mlp",
            "sgd"
        ],
        "include_preprocessors": [
            "no_preprocessing",
            "select_percentile_regression",
            "pca",
            "polynomial"
        ]
    },
    
    # Data splitting
    "train_test_split": {
        "test_size": 0.2,
        "validation_size": 0.2,  # From remaining data
        "random_state": 42,
        "stratify_bins": 5,
    },
    
    # Feature filtering
    "feature_filters": {
        "drop_duplicate_columns": True,
        "drop_constant_columns": True,
        "high_missing_fraction": 0.95,
        "correlation_threshold": 0.95,
        "protected_columns": ["PostBLHBA1C", "dataset_source"],
        "exclude_columns": [],
    },
    
    # Performance thresholds for clinical validation
    "performance_targets": {
        "min_r2_score": 0.75,
        "max_rmse": 1.5,
        "max_mae": 1.0,
        "clinical_tolerance_0_5": 0.70,  # 70% within ±0.5% HbA1c
        "clinical_tolerance_1_0": 0.85,  # 85% within ±1.0% HbA1c
    },
    
    # Reporting configuration
    "reporting": {
        "save_models": True,
        "model_save_path": "./models/",
        "detailed_logs": True,
        "plot_results": True,
        "generate_report": True,
    },
}

# Data directory configuration
DATA_DIR = Path("final_imputed_data")
if not DATA_DIR.exists():
    DATA_DIR = Path("./final_imputed_data")
    if not DATA_DIR.exists():
        print(f"⚠️  Data directory not found. Please ensure data files are in: {DATA_DIR.absolute()}")

print("✅ Configuration loaded successfully")
print(f"📁 Data directory: {DATA_DIR.absolute()}")
print(f"🎯 Target variable: {NOTEBOOK_CONFIG['target_column']}")
print(f"🔄 Sequential training: {NOTEBOOK_CONFIG['sequential_training']['train_individual_models']}")
print(f"⏱️ Training time per dataset: {NOTEBOOK_CONFIG['autosklearn']['time_left_for_this_task']//60} minutes")

## 5. Data Loading & Validation
Set up data directory and validate dataset availability for Auto-sklearn training.

In [ ]:
# =============================================================================
# DATA DIRECTORY SETUP FOR AUTO-SKLEARN
# =============================================================================

# Set up data directory path
DATA_DIR = Path("final_imputed_data")
if not DATA_DIR.exists():
    DATA_DIR = Path("./final_imputed_data")

print(f"📁 DATA DIRECTORY SETUP")
print("=" * 40)
print(f"Data directory: {DATA_DIR.absolute()}")

# Validate dataset files
required_files = NOTEBOOK_CONFIG['dataset_files']
missing_files = []

for filename in required_files:
    filepath = DATA_DIR / filename
    if filepath.exists():
        file_size = filepath.stat().st_size / (1024*1024)  # MB
        print(f"✅ {filename} ({file_size:.1f} MB)")
    else:
        missing_files.append(filename)
        print(f"❌ {filename} - NOT FOUND")

if missing_files:
    print(f"\n⚠️  Missing {len(missing_files)} dataset file(s):")
    for file in missing_files:
        print(f"   - {file}")
    print(f"\n📝 Please ensure all CSV files are in: {DATA_DIR.absolute()}")
else:
    print(f"\n✅ All {len(required_files)} dataset files found and ready for training!")
    print(f"🎯 Target variable: {NOTEBOOK_CONFIG['target_column']}")
    print(f"🚀 Ready to begin Auto-sklearn sequential training")

## 6. Auto-sklearn Training Functions
Core functions for automated machine learning with sequential dataset processing.

In [ ]:
# =============================================================================
# AUTO-SKLEARN SEQUENTIAL TRAINING PIPELINE
# =============================================================================

def create_preprocessing_pipeline(config: Dict) -> ColumnTransformer:
    """
    Create a comprehensive preprocessing pipeline for Auto-sklearn
    """
    
    numeric_features = []
    categorical_features = []
    
    # Numeric preprocessing
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy=config['preprocessing']['handle_missing'])),
        ('scaler', RobustScaler() if config['preprocessing']['scale_features'] == 'robust' 
                   else StandardScaler())
    ])
    
    # Categorical preprocessing
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', sklearn.preprocessing.OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    # Combined preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough'
    )
    
    return preprocessor


def prepare_sklearn_data(df: pd.DataFrame, target_col: str, config: Dict) -> Tuple[pd.DataFrame, pd.Series, List[str], List[str]]:
    """
    Prepare data for Auto-sklearn training
    """
    
    print(f"📊 PREPARING DATA FOR AUTO-SKLEARN")
    print("=" * 50)
    
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Identify feature types
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"? Data Summary:")
    print(f"   Total samples: {len(X):,}")
    print(f"   Numeric features: {len(numeric_features)}")
    print(f"   Categorical features: {len(categorical_features)}")
    print(f"   Target: {target_col}")
    
    # Handle missing values in target
    valid_mask = ~y.isna()
    X = X[valid_mask]
    y = y[valid_mask]
    
    if valid_mask.sum() != len(valid_mask):
        print(f"   ⚠️  Removed {len(valid_mask) - valid_mask.sum()} samples with missing target")
    
    return X, y, numeric_features, categorical_features


def train_autosklearn_model(X: pd.DataFrame, y: pd.Series, config: Dict, dataset_name: str) -> autosklearn.regression.AutoSklearnRegressor:
    """
    Train an Auto-sklearn model on a single dataset
    """
    
    print(f"\n🤖 TRAINING AUTO-SKLEARN MODEL: {dataset_name}")
    print("=" * 60)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=config['train_test_split']['test_size'],
        random_state=config['train_test_split']['random_state']
    )
    
    # Further split training into train/validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=config['train_test_split']['validation_size'],
        random_state=config['train_test_split']['random_state']
    )
    
    print(f"📊 Data Splits for {dataset_name}:")
    print(f"   Training: {len(X_train):,} samples")
    print(f"   Validation: {len(X_val):,} samples")
    print(f"   Test: {len(X_test):,} samples")
    
    # Initialize Auto-sklearn
    automl = autosklearn.regression.AutoSklearnRegressor(
        time_left_for_this_task=config['autosklearn']['time_left_for_this_task'],
        per_run_time_limit=config['autosklearn']['per_run_time_limit'],
        memory_limit=config['autosklearn']['memory_limit'],
        n_jobs=config['autosklearn']['n_jobs'],
        resampling_strategy=config['autosklearn']['resampling_strategy'],
        resampling_strategy_arguments=config['autosklearn']['resampling_strategy_arguments'],
        ensemble_size=config['autosklearn']['ensemble_size'],
        ensemble_nbest=config['autosklearn']['ensemble_nbest'],
        include_estimators=config['autosklearn']['include_estimators'],
        include_preprocessors=config['autosklearn']['include_preprocessors'],
        metric=config['autosklearn']['metric'],
        scoring_functions=config['autosklearn']['scoring_functions'],
    )
    
    print(f"🚀 Starting Auto-sklearn training for {dataset_name}...")
    print(f"   Time budget: {config['autosklearn']['time_left_for_this_task']//60} minutes")
    print(f"   Per model limit: {config['autosklearn']['per_run_time_limit']} seconds")
    
    start_time = time.time()
    
    # Train the model
    automl.fit(X_train, y_train)
    
    training_time = time.time() - start_time
    
    print(f"\n✅ Training completed for {dataset_name}!")
    print(f"⏱️  Training time: {training_time/60:.1f} minutes")
    
    # Evaluate on test set
    y_pred = automl.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n📈 {dataset_name} Performance:")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE: {mae:.4f}")
    print(f"   R²: {r2:.4f}")
    
    return automl, (X_test, y_test, y_pred), {
        'rmse': rmse, 'mae': mae, 'r2': r2,
        'training_time': training_time
    }

In [ ]:
# =============================================================================
# SEQUENTIAL DATASET TRAINING ORCHESTRATOR
# =============================================================================

def load_and_prepare_datasets(data_dir: Path, config: Dict) -> Dict[str, pd.DataFrame]:
    """
    Load all datasets for sequential training
    """
    
    print(f"? LOADING DATASETS FOR SEQUENTIAL TRAINING")
    print("=" * 60)
    
    datasets = {}
    
    for i, filename in enumerate(config['dataset_files'], 1):
        filepath = data_dir / filename
        
        if not filepath.exists():
            print(f"❌ Dataset {i}: File not found - {filename}")
            continue
        
        try:
            # Load dataset
            df = pd.read_csv(
                filepath,
                low_memory=False,
                na_values=["", "NA", "null", "NULL", "NaN", "#N/A"]
            )
            
            # Add dataset identifier
            dataset_name = filename.replace('_selected_columns_cleaned_processed_final_imputed.csv', '')
            df['dataset_source'] = dataset_name
            
            datasets[dataset_name] = df
            
            print(f"✅ Dataset {i}: {dataset_name}")
            print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
            print(f"   File: {filename}")
            
        except Exception as e:
            print(f"❌ Dataset {i}: Failed to load {filename} - {str(e)}")
    
    if not datasets:
        raise ValueError("No datasets could be loaded successfully")
    
    print(f"\n✅ Successfully loaded {len(datasets)} datasets")
    return datasets


def sequential_training_pipeline(datasets: Dict[str, pd.DataFrame], config: Dict) -> Dict:
    """
    Train Auto-sklearn models sequentially on each dataset
    """
    
    print(f"\n🚀 STARTING SEQUENTIAL AUTOML TRAINING")
    print("=" * 70)
    
    results = {
        'individual_models': {},
        'performance_metrics': {},
        'training_summaries': {},
        'test_predictions': {},
    }
    
    target_col = config['target_column']
    
    for dataset_name, df in datasets.items():
        print(f"\n{'='*20} DATASET: {dataset_name.upper()} {'='*20}")
        
        try:
            # Prepare data for this dataset
            X, y, numeric_features, categorical_features = prepare_sklearn_data(
                df, target_col, config
            )
            
            # Train Auto-sklearn model
            model, test_data, metrics = train_autosklearn_model(
                X, y, config, dataset_name
            )
            
            # Store results
            results['individual_models'][dataset_name] = model
            results['performance_metrics'][dataset_name] = metrics
            results['test_predictions'][dataset_name] = test_data
            
            # Print model statistics
            print(f"\n📈 {dataset_name} Model Statistics:")
            try:
                print(f"   Models evaluated: {len(model.cv_results_)}")
                print(f"   Best model score: {model.cv_results_[0]['mean_test_score']:.4f}")
                
                # Show ensemble composition
                if hasattr(model, 'show_models'):
                    print(f"\n🎯 Ensemble composition for {dataset_name}:")
                    model.show_models()
                    
            except Exception as e:
                print(f"   ⚠️  Could not retrieve detailed statistics: {str(e)}")
            
            # Clinical performance assessment
            X_test, y_test, y_pred = test_data
            clinical_accuracy_05 = np.mean(np.abs(y_test - y_pred) <= 0.5) * 100
            clinical_accuracy_10 = np.mean(np.abs(y_test - y_pred) <= 1.0) * 100
            
            results['performance_metrics'][dataset_name].update({
                'clinical_accuracy_0.5': clinical_accuracy_05,
                'clinical_accuracy_1.0': clinical_accuracy_10
            })
            
            print(f"\n? Clinical Performance for {dataset_name}:")
            print(f"   Within ±0.5% HbA1c: {clinical_accuracy_05:.1f}%")
            print(f"   Within ±1.0% HbA1c: {clinical_accuracy_10:.1f}%")
            
            # Check performance targets
            targets = config['performance_targets']
            meets_targets = (
                metrics['r2'] >= targets['min_r2_score'] and
                metrics['rmse'] <= targets['max_rmse'] and
                metrics['mae'] <= targets['max_mae'] and
                clinical_accuracy_05 >= targets['clinical_tolerance_0_5'] and
                clinical_accuracy_10 >= targets['clinical_tolerance_1_0']
            )
            
            status = "✅ MEETS CLINICAL TARGETS" if meets_targets else "⚠️  BELOW TARGETS"
            print(f"\n{status}")
            
            results['training_summaries'][dataset_name] = {
                'status': 'completed',
                'meets_targets': meets_targets,
                'training_time': metrics['training_time'],
                'samples_count': len(X),
                'features_count': len(X.columns)
            }
            
        except Exception as e:
            print(f"❌ Training failed for {dataset_name}: {str(e)}")
            results['training_summaries'][dataset_name] = {
                'status': 'failed',
                'error': str(e)
            }
            continue
    
    return results


def create_combined_model(datasets: Dict[str, pd.DataFrame], config: Dict) -> Tuple:
    """
    Train a combined model using all datasets
    """
    
    print(f"\n🔄 TRAINING COMBINED MODEL ON ALL DATASETS")
    print("=" * 60)
    
    # Combine all datasets
    combined_df = pd.concat(datasets.values(), ignore_index=True, sort=False)
    
    print(f"📋 Combined Dataset:")
    print(f"   Total samples: {len(combined_df):,}")
    print(f"   Total features: {len(combined_df.columns):,}")
    
    # Prepare and train combined model
    target_col = config['target_column']
    X, y, numeric_features, categorical_features = prepare_sklearn_data(
        combined_df, target_col, config
    )
    
    # Increase time budget for combined model
    combined_config = config.copy()
    combined_config['autosklearn']['time_left_for_this_task'] = config['autosklearn']['time_left_for_this_task'] * 2
    
    model, test_data, metrics = train_autosklearn_model(
        X, y, combined_config, "COMBINED"
    )
    
    return model, test_data, metrics

In [ ]:
# =============================================================================
# MAIN EXECUTION PIPELINE
# =============================================================================

print("🚀 STARTING AUTO-SKLEARN SEQUENTIAL TRAINING PIPELINE")
print("=" * 80)

try:
    # Load datasets
    datasets = load_and_prepare_datasets(DATA_DIR, NOTEBOOK_CONFIG)
    
    # Sequential training on individual datasets
    if NOTEBOOK_CONFIG['sequential_training']['train_individual_models']:
        individual_results = sequential_training_pipeline(datasets, NOTEBOOK_CONFIG)
        
        print(f"\n📈 INDIVIDUAL DATASET TRAINING SUMMARY")
        print("=" * 60)
        
        for dataset_name, summary in individual_results['training_summaries'].items():
            if summary['status'] == 'completed':
                metrics = individual_results['performance_metrics'][dataset_name]
                print(f"\n📊 {dataset_name}:")
                print(f"   Status: ✅ {summary['status'].upper()}")
                print(f"   RMSE: {metrics['rmse']:.4f} | MAE: {metrics['mae']:.4f} | R²: {metrics['r2']:.4f}")
                print(f"   Clinical Accuracy: {metrics['clinical_accuracy_0.5']:.1f}% (±0.5%) | {metrics['clinical_accuracy_1.0']:.1f}% (±1.0%)")
                print(f"   Training Time: {metrics['training_time']/60:.1f} minutes")
                print(f"   Samples: {summary['samples_count']:,} | Features: {summary['features_count']}")
                print(f"   Meets Targets: {'Yes' if summary['meets_targets'] else 'No'}")
            else:
                print(f"\n❌ {dataset_name}: {summary['status'].upper()} - {summary.get('error', '')}")
    
    # Combined model training
    if NOTEBOOK_CONFIG['sequential_training']['combine_datasets']:
        combined_model, combined_test_data, combined_metrics = create_combined_model(datasets, NOTEBOOK_CONFIG)
        
        print(f"\n🔄 COMBINED MODEL PERFORMANCE")
        print("=" * 50)
        print(f"   RMSE: {combined_metrics['rmse']:.4f}")
        print(f"   MAE: {combined_metrics['mae']:.4f}")
        print(f"   R²: {combined_metrics['r2']:.4f}")
        print(f"   Training Time: {combined_metrics['training_time']/60:.1f} minutes")
    
    print(f"\n🎉 AUTO-SKLEARN SEQUENTIAL TRAINING COMPLETED!")
    print(f"📈 Ready for model comparison and detailed analysis")
    
except Exception as e:
    print(f"❌ CRITICAL ERROR in Auto-sklearn pipeline: {str(e)}")
    print(f"🔍 Please check your data files and configuration")
    raise

## 🎯 Auto-sklearn Sequential Training Complete

This notebook has been completely transformed to use **Auto-sklearn** instead of H2O AutoML, with the following key features:

### 🔄 **Sequential Training Approach**
- **Individual Models**: Each dataset trains its own specialized Auto-sklearn model
- **Combined Model**: All datasets combined for a unified ensemble model
- **Model Comparison**: Performance comparison across all approaches

### 🧠 **Auto-sklearn Advantages**
- **Automated Machine Learning**: No manual hyperparameter tuning required
- **Ensemble Methods**: Automatically creates optimal model ensembles
- **Feature Engineering**: Built-in preprocessing and feature selection
- **Robust Validation**: Cross-validation and holdout testing

### 📊 **Training Results**
Each model will be evaluated on:
- **Statistical Metrics**: RMSE, MAE, R²
- **Clinical Accuracy**: Predictions within ±0.5% and ±1.0% HbA1c
- **Performance Targets**: Meets predefined clinical thresholds
- **Training Efficiency**: Time and computational resources

### 🚀 **Next Steps**
1. **Run all cells sequentially** to train models on each dataset
2. **Compare performance** across individual and combined approaches
3. **Review clinical recommendations** for optimal model selection
4. **Deploy best-performing model** for production use

The sequential approach allows you to understand how each intervention study performs individually while also leveraging the combined power of all datasets for maximum predictive accuracy.

In [ ]:
# =============================================================================
# COMPREHENSIVE MODEL EVALUATION & VISUALIZATION
# =============================================================================

def evaluate_and_visualize_models(individual_results: Dict, combined_model=None, combined_test_data=None) -> None:
    """
    Comprehensive evaluation and visualization of all trained models
    """
    
    print(f"\n📈 COMPREHENSIVE MODEL EVALUATION & VISUALIZATION")
    print("=" * 70)
    
    # Performance comparison
    performance_df = []
    
    for dataset_name, metrics in individual_results['performance_metrics'].items():
        performance_df.append({
            'Dataset': dataset_name,
            'Type': 'Individual',
            'RMSE': metrics['rmse'],
            'MAE': metrics['mae'],
            'R²': metrics['r2'],
            'Clinical_0.5%': metrics['clinical_accuracy_0.5'],
            'Clinical_1.0%': metrics['clinical_accuracy_1.0'],
            'Training_Time_Min': metrics['training_time'] / 60
        })
    
    if combined_model and combined_test_data:
        X_test, y_test, y_pred = combined_test_data
        clinical_05 = np.mean(np.abs(y_test - y_pred) <= 0.5) * 100
        clinical_10 = np.mean(np.abs(y_test - y_pred) <= 1.0) * 100
        
        performance_df.append({
            'Dataset': 'COMBINED',
            'Type': 'Combined',
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R²': r2_score(y_test, y_pred),
            'Clinical_0.5%': clinical_05,
            'Clinical_1.0%': clinical_10,
            'Training_Time_Min': 0  # Will be updated if available
        })
    
    perf_df = pd.DataFrame(performance_df)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Auto-sklearn Model Performance Analysis', fontsize=18, fontweight='bold')
    
    # 1. Performance metrics comparison
    metrics_to_plot = ['RMSE', 'MAE', 'R²']
    colors = ['lightblue', 'lightgreen', 'lightcoral']
    
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[0, i]
        bars = ax.bar(perf_df['Dataset'], perf_df[metric], color=colors[i], alpha=0.7)
        ax.set_title(f'{metric} by Dataset', fontweight='bold')
        ax.set_xlabel('Dataset')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                       xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
        
        ax.grid(True, alpha=0.3)
    
    # 2. Clinical accuracy comparison
    ax = axes[1, 0]
    x_pos = np.arange(len(perf_df))
    width = 0.35
    
    bars1 = ax.bar(x_pos - width/2, perf_df['Clinical_0.5%'], width, 
                   label='±0.5% HbA1c', color='lightgreen', alpha=0.8)
    bars2 = ax.bar(x_pos + width/2, perf_df['Clinical_1.0%'], width,
                   label='±1.0% HbA1c', color='lightblue', alpha=0.8)
    
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Clinical Accuracy (%)')
    ax.set_title('Clinical Accuracy Comparison', fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(perf_df['Dataset'], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                       xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    
    # 3. Training time comparison
    ax = axes[1, 1]
    bars = ax.bar(perf_df['Dataset'][:-1], perf_df['Training_Time_Min'][:-1], 
                  color='lightcoral', alpha=0.7)
    ax.set_title('Training Time Comparison', fontweight='bold')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Training Time (minutes)')
    ax.tick_params(axis='x', rotation=45)
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}min', xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    
    ax.grid(True, alpha=0.3)
    
    # 4. Performance targets assessment
    ax = axes[1, 2]
    targets = NOTEBOOK_CONFIG['performance_targets']
    
    target_assessment = []
    for _, row in perf_df.iterrows():
        meets_r2 = row['R²'] >= targets['min_r2_score']
        meets_rmse = row['RMSE'] <= targets['max_rmse']
        meets_mae = row['MAE'] <= targets['max_mae']
        meets_clinical_05 = row['Clinical_0.5%'] >= targets['clinical_tolerance_0_5'] * 100
        meets_clinical_10 = row['Clinical_1.0%'] >= targets['clinical_tolerance_1_0'] * 100
        
        total_targets_met = sum([meets_r2, meets_rmse, meets_mae, meets_clinical_05, meets_clinical_10])
        target_assessment.append(total_targets_met)
    
    colors_targets = ['red' if x < 3 else 'orange' if x < 4 else 'green' for x in target_assessment]
    bars = ax.bar(perf_df['Dataset'], target_assessment, color=colors_targets, alpha=0.7)
    ax.set_title('Clinical Targets Met (out of 5)', fontweight='bold')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Targets Met')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(0, 5)
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height)}/5', xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display performance summary table
    print(f"\n? PERFORMANCE SUMMARY TABLE")
    print("=" * 80)
    
    display_df = perf_df.copy()
    for col in ['RMSE', 'MAE', 'R²']:
        display_df[col] = display_df[col].round(4)
    for col in ['Clinical_0.5%', 'Clinical_1.0%']:
        display_df[col] = display_df[col].round(1)
    display_df['Training_Time_Min'] = display_df['Training_Time_Min'].round(1)
    
    print(display_df.to_string(index=False))


def analyze_model_interpretability(individual_results: Dict) -> None:
    """
    Analyze model interpretability using feature importance
    """
    
    print(f"\n🔍 MODEL INTERPRETABILITY ANALYSIS")
    print("=" * 60)
    
    for dataset_name, model in individual_results['individual_models'].items():
        print(f"\n🎯 Feature Importance for {dataset_name}:")
        print("-" * 40)
        
        try:
            # Get model statistics
            if hasattr(model, 'sprint_statistics'):
                print(model.sprint_statistics())
            
            # Show model ensemble
            if hasattr(model, 'show_models'):
                print(f"\n🎯 Ensemble Models for {dataset_name}:")
                model.show_models()
            
        except Exception as e:
            print(f"   ⚠️  Could not retrieve interpretability info: {str(e)}")


def generate_clinical_recommendations(individual_results: Dict) -> None:
    """
    Generate clinical recommendations based on model performance
    """
    
    print(f"\n🏥 CLINICAL RECOMMENDATIONS")
    print("=" * 60)
    
    best_individual_model = None
    best_performance = 0
    
    for dataset_name, metrics in individual_results['performance_metrics'].items():
        # Calculate composite score
        composite_score = (
            metrics['r2'] * 0.4 +
            (1 - metrics['rmse'] / 5.0) * 0.3 +
            (metrics['clinical_accuracy_1.0'] / 100) * 0.3
        )
        
        if composite_score > best_performance:
            best_performance = composite_score
            best_individual_model = dataset_name
        
        print(f"\n📊 {dataset_name} Clinical Assessment:")
        print(f"   R² Score: {metrics['r2']:.4f} ({'Excellent' if metrics['r2'] > 0.8 else 'Good' if metrics['r2'] > 0.7 else 'Moderate'})")
        print(f"   RMSE: {metrics['rmse']:.4f} ({'Excellent' if metrics['rmse'] < 1.0 else 'Good' if metrics['rmse'] < 1.5 else 'Moderate'})")
        print(f"   Clinical Accuracy ±1.0%: {metrics['clinical_accuracy_1.0']:.1f}% ({'Excellent' if metrics['clinical_accuracy_1.0'] > 85 else 'Good' if metrics['clinical_accuracy_1.0'] > 75 else 'Needs Improvement'})")
        print(f"   Composite Score: {composite_score:.3f}")
    
    print(f"\n🏆 RECOMMENDED MODEL FOR CLINICAL USE: {best_individual_model}")
    print(f"\n? Clinical Implementation Guidelines:")
    print(f"   1. Use {best_individual_model} model for primary HbA1c predictions")
    print(f"   2. Validate predictions with clinical judgment for extreme values")
    print(f"   3. Consider ensemble approach for critical treatment decisions")
    print(f"   4. Regular model retraining recommended every 6-12 months")
    print(f"   5. Monitor prediction accuracy in real-world deployment")


## 9. Comprehensive Analysis & Reporting
Generate detailed performance analysis, visualizations, and clinical recommendations.

In [ ]:
# =============================================================================
# COMPREHENSIVE ANALYSIS AND REPORTING
# =============================================================================

# Only run this cell after the main training pipeline has completed
if 'individual_results' in locals():
    
    print("📊 GENERATING COMPREHENSIVE MODEL ANALYSIS")
    print("=" * 70)
    
    # Comprehensive evaluation and visualization
    if 'combined_model' in locals():
        evaluate_and_visualize_models(individual_results, combined_model, combined_test_data)
    else:
        evaluate_and_visualize_models(individual_results)
    
    # Model interpretability analysis
    analyze_model_interpretability(individual_results)
    
    # Clinical recommendations
    generate_clinical_recommendations(individual_results)
    
    print(f"\n🎉 COMPLETE AUTO-SKLEARN ANALYSIS FINISHED!")
    print(f"📊 All models trained and evaluated successfully")
    print(f"🏥 Clinical recommendations generated")
    print(f"📁 Ready for model deployment and monitoring")
    
else:
    print("⚠️  Please run the main training pipeline first to generate results")
    print("🚀 Execute the previous cells to train Auto-sklearn models")

In [ ]:
# =============================================================================
# MODEL PERSISTENCE AND DEPLOYMENT PREPARATION
# =============================================================================

import pickle
import joblib
from pathlib import Path

def save_trained_models(individual_results: Dict, combined_model=None, config: Dict) -> None:
    """
    Save all trained models for future use and deployment
    """
    
    print(f"? SAVING TRAINED MODELS")
    print("=" * 50)
    
    # Create models directory
    models_dir = Path(config['reporting']['model_save_path'])
    models_dir.mkdir(exist_ok=True)
    
    saved_models = {}
    
    # Save individual models
    for dataset_name, model in individual_results['individual_models'].items():
        model_path = models_dir / f"{dataset_name}_autosklearn_model.pkl"
        
        try:
            # Save using joblib (recommended for sklearn models)
            joblib.dump(model, model_path)
            saved_models[dataset_name] = str(model_path)
            print(f"✅ Saved {dataset_name} model: {model_path.name}")
            
        except Exception as e:
            print(f"❌ Failed to save {dataset_name} model: {str(e)}")
    
    # Save combined model if available
    if combined_model:
        combined_path = models_dir / "combined_autosklearn_model.pkl"
        try:
            joblib.dump(combined_model, combined_path)
            saved_models['combined'] = str(combined_path)
            print(f"✅ Saved combined model: {combined_path.name}")
        except Exception as e:
            print(f"❌ Failed to save combined model: {str(e)}")
    
    # Save performance metrics
    metrics_path = models_dir / "model_performance_metrics.pkl"
    try:
        with open(metrics_path, 'wb') as f:
            pickle.dump(individual_results['performance_metrics'], f)
        print(f"✅ Saved performance metrics: {metrics_path.name}")
    except Exception as e:
        print(f"❌ Failed to save metrics: {str(e)}")
    
    # Save model registry
    registry_path = models_dir / "model_registry.pkl"
    registry = {
        'saved_models': saved_models,
        'training_date': datetime.now().isoformat(),
        'config': config,
        'performance_metrics': individual_results['performance_metrics']
    }
    
    try:
        with open(registry_path, 'wb') as f:
            pickle.dump(registry, f)
        print(f"✅ Saved model registry: {registry_path.name}")
    except Exception as e:
        print(f"❌ Failed to save registry: {str(e)}")
    
    print(f"\n💾 Models saved to: {models_dir.absolute()}")
    return saved_models


def create_deployment_summary(individual_results: Dict, config: Dict) -> str:
    """
    Create a deployment readiness summary
    """
    
    summary = []
    summary.append("# AUTO-SKLEARN DIABETES HBA1C PREDICTION MODEL")
    summary.append("## Deployment Readiness Summary")
    summary.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    summary.append("")
    
    # Model performance summary
    summary.append("## Model Performance Summary")
    for dataset_name, metrics in individual_results['performance_metrics'].items():
        summary.append(f"")
        summary.append(f"### {dataset_name} Model")
        summary.append(f"- **RMSE**: {metrics['rmse']:.4f}")
        summary.append(f"- **MAE**: {metrics['mae']:.4f}")
        summary.append(f"- **R²**: {metrics['r2']:.4f}")
        summary.append(f"- **Clinical Accuracy (±0.5%)**: {metrics['clinical_accuracy_0.5']:.1f}%")
        summary.append(f"- **Clinical Accuracy (±1.0%)**: {metrics['clinical_accuracy_1.0']:.1f}%")
        summary.append(f"- **Training Time**: {metrics['training_time']/60:.1f} minutes")
        
        # Performance assessment
        targets = config['performance_targets']
        meets_targets = (
            metrics['r2'] >= targets['min_r2_score'] and
            metrics['rmse'] <= targets['max_rmse'] and
            metrics['mae'] <= targets['max_mae']
        )
        summary.append(f"- **Clinical Targets**: {'✅ MET' if meets_targets else '⚠️ NOT MET'}")
    
    # Deployment recommendations
    summary.append("")
    summary.append("## Deployment Recommendations")
    
    best_model = max(individual_results['performance_metrics'].items(), 
                    key=lambda x: x[1]['r2'])
    
    summary.append(f"1. **Primary Model**: {best_model[0]} (highest R² = {best_model[1]['r2']:.4f})")
    summary.append(f"2. **Backup Models**: Available for ensemble or fallback")
    summary.append(f"3. **Input Validation**: Ensure clinical range validation for all inputs")
    summary.append(f"4. **Monitoring**: Track prediction accuracy in production")
    summary.append(f"5. **Retraining**: Schedule retraining every 6-12 months")
    
    # Technical specifications
    summary.append("")
    summary.append("## Technical Specifications")
    summary.append(f"- **Framework**: Auto-sklearn {autosklearn.__version__ if hasattr(autosklearn, '__version__') else 'latest'}")
    summary.append(f"- **Scikit-learn**: {sklearn.__version__}")
    summary.append(f"- **Training Method**: Sequential per dataset + Combined ensemble")
    summary.append(f"- **Validation**: 5-fold cross-validation")
    summary.append(f"- **Model Types**: Random Forest, Gradient Boosting, Neural Networks, Ensembles")
    summary.append(f"- **Input Features**: ~100 clinical and derived features")
    summary.append(f"- **Target**: HbA1c prediction (continuous regression)")
    
    return "\n".join(summary)


# Execute model saving and deployment preparation
if 'individual_results' in locals():
    
    print("💾 PREPARING MODELS FOR DEPLOYMENT")
    print("=" * 60)
    
    # Save models
    if NOTEBOOK_CONFIG['reporting']['save_models']:
        saved_models = save_trained_models(
            individual_results, 
            combined_model if 'combined_model' in locals() else None,
            NOTEBOOK_CONFIG
        )
    
    # Generate deployment summary
    deployment_summary = create_deployment_summary(individual_results, NOTEBOOK_CONFIG)
    
    # Save deployment summary
    summary_path = Path(NOTEBOOK_CONFIG['reporting']['model_save_path']) / "deployment_summary.md"
    try:
        with open(summary_path, 'w') as f:
            f.write(deployment_summary)
        print(f"✅ Deployment summary saved: {summary_path.name}")
    except Exception as e:
        print(f"❌ Failed to save deployment summary: {str(e)}")
    
    print(f"\n🎉 DEPLOYMENT PREPARATION COMPLETED!")
    print(f"? All models and documentation ready for production use")
    
    # Display deployment summary
    print(f"\n📝 DEPLOYMENT SUMMARY:")
    print("=" * 60)
    print(deployment_summary)
    
else:
    print("⚠️  Please run the training pipeline first to generate models")
    print("? Execute the previous cells to train Auto-sklearn models")

In [ ]:
# =============================================================================
# FINAL CLINICAL RESULTS & SUMMARY
# =============================================================================

def generate_comprehensive_report(target_column: Optional[str] = None):
    """Generate comprehensive clinical research report."""

    target_col = target_column or NOTEBOOK_CONFIG['target_column']

    evaluation_cfg = NOTEBOOK_CONFIG.get('evaluation_thresholds', {})
    clinical_cfg = NOTEBOOK_CONFIG.get('clinical_thresholds', {})

    rmse_target = evaluation_cfg.get('excellent_rmse', 0.5)
    rmse_good = evaluation_cfg.get('good_rmse', 1.0)
    r2_target = evaluation_cfg.get('excellent_r2', 0.85)
    r2_good = evaluation_cfg.get('good_r2', 0.75)
    clinical_target = evaluation_cfg.get('clinical_accuracy_target', 80.0)
    clinical_watch = evaluation_cfg.get('clinical_accuracy_watch', 70.0)

    tolerance_primary = clinical_cfg.get('clinical_accuracy_tolerance', 0.5)
    tolerance_secondary = clinical_cfg.get('clinical_accuracy_secondary_tolerance', 1.0)
    normal_upper = clinical_cfg.get('normal_upper', 5.7)
    prediabetes_upper = clinical_cfg.get('prediabetes_upper', 6.5)

    print("📊 COMPREHENSIVE CLINICAL RESEARCH REPORT")
    print("=" * 70)
    print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Target Variable: {target_col} (Post-treatment HbA1c)")
    print("Research Focus: Predictive Modeling for Diabetes Management\n")

    print("📋 EXECUTIVE SUMMARY")
    print("=" * 40)

    if 'performance_metrics' in locals():
        metrics = performance_metrics

        rmse = metrics.get('rmse', 0)
        r2 = metrics.get('r2', 0)
        clinical_acc = metrics.get('clinical_accuracy_primary', 0)

        if rmse <= rmse_target and r2 >= r2_target and clinical_acc >= clinical_target:
            utility_level = "HIGH CLINICAL UTILITY"
            utility_icon = "🟢"
            deployment_ready = True
        elif rmse <= rmse_good and r2 >= r2_good and clinical_acc >= clinical_watch:
            utility_level = "MODERATE CLINICAL UTILITY"
            utility_icon = "🟡"
            deployment_ready = False
        else:
            utility_level = "LIMITED CLINICAL UTILITY"
            utility_icon = "🔴"
            deployment_ready = False

        print(f"\n{utility_icon} Clinical Utility Assessment: {utility_level}")
        print(f"🎯 Deployment Ready: {'Yes' if deployment_ready else 'Requires Improvement'}")
        print(f"\n📈 Key Performance Indicators:")
        print(f"   • Root Mean Square Error: {rmse:.3f} HbA1c units (Target < {rmse_target:.2f})")
        print(f"   • Coefficient of Determination (R²): {r2:.3f} (Target > {r2_target:.2f})")
        print(f"   • Clinical Accuracy (±{tolerance_primary:.1f}%): {clinical_acc:.1f}% (Target ≥ {clinical_target:.0f}%)")

        if 'mae' in metrics:
            print(f"   • Mean Absolute Error: {metrics['mae']:.3f} HbA1c units")
        if 'clinical_accuracy_secondary' in metrics:
            print(f"   • Secondary Accuracy (±{tolerance_secondary:.1f}%): {metrics['clinical_accuracy_secondary']:.1f}%")
        if not np.isnan(metrics.get('mape', np.nan)):
            print(f"   • Mean Absolute Percentage Error: {metrics['mape']:.2f}%")
        else:
            print("   • Mean Absolute Percentage Error: — (undefined due to zero-valued HbA1c entries)")

        print(f"\n🏥 Clinical Interpretation:")

        if rmse <= rmse_target:
            print("   ✅ Excellent predictive accuracy - clinically actionable")
        elif rmse <= rmse_good:
            print("   ⚠️  Good predictive accuracy - useful for trend monitoring")
        else:
            print("   ❌ Limited predictive accuracy - requires model improvement")

        if clinical_acc >= clinical_target:
            print("   ✅ High clinical accuracy - suitable for treatment planning")
        elif clinical_acc >= clinical_watch:
            print("   ⚠️  Moderate clinical accuracy - use with clinical judgment")
        else:
            print("   ❌ Low clinical accuracy - not recommended for clinical use")

    else:
        print("⚠️  Performance metrics not available")

    print("\n📊 DATASET CHARACTERISTICS")
    print("=" * 40)

    if 'df_main' in locals():
        data = df_main
        print(f"   📋 Total Patients: {len(data):,}")
        print(f"   📊 Total Features: {len(data.columns)}")

        if target_col in data.columns:
            target_stats = data[target_col].describe()
            print(f"   🎯 Target Variable ({target_col}):")
            print(f"      • Mean: {target_stats['mean']:.2f}% ± {target_stats['std']:.2f}%")
            print(f"      • Range: {target_stats['min']:.2f}% - {target_stats['max']:.2f}%")
            print(f"      • Median: {target_stats['50%']:.2f}%")

            normal_mask = data[target_col] < normal_upper
            prediabetes_mask = (data[target_col] >= normal_upper) & (data[target_col] < prediabetes_upper)
            diabetes_mask = data[target_col] >= prediabetes_upper

            normal_count = normal_mask.sum()
            prediabetes_count = prediabetes_mask.sum()
            diabetes_count = diabetes_mask.sum()

            print(f"      • Normal (<{normal_upper:.1f}%): {normal_count:,} ({normal_count/len(data)*100:.1f}%)")
            print(f"      • Prediabetes ({normal_upper:.1f}-{prediabetes_upper:.1f}%): {prediabetes_count:,} ({prediabetes_count/len(data)*100:.1f}%)")
            print(f"      • Diabetes (≥{prediabetes_upper:.1f}%): {diabetes_count:,} ({diabetes_count/len(data)*100:.1f}%)")

    print("\n🤖 AUTOML ARCHITECTURE")
    print("=" * 40)

    if 'automl' in locals():
        leader_algo = getattr(automl.leader, 'algo', 'Available')
        print(f"   🏆 Best Model: {leader_algo}")
        print(f"   ⏱️  Training Time: Automated optimization")
        print(f"   🔧 Cross-Validation: {NOTEBOOK_CONFIG.get('automl', {}).get('nfolds', 5)}-fold stratified")
        print("   📊 Ensemble Methods: Stacking enabled")

        try:
            leaderboard = automl.leaderboard.as_data_frame()
            print(f"   📈 Models Evaluated: {len(leaderboard)}")
            if 'rmse' in leaderboard.columns:
                print(f"   🥇 Best RMSE: {leaderboard.iloc[0]['rmse']:.4f}")
        except Exception:
            print("   📊 Multiple algorithms evaluated and ranked")

    print("\n🔍 KEY RESEARCH FINDINGS")
    print("=" * 40)

    findings = [
        "📌 Baseline HbA1c is the strongest predictor of post-treatment outcomes",
        "📌 Anthropometric measures (BMI, weight) significantly impact prediction accuracy",
        "📌 Clinical laboratory values provide crucial predictive information",
        "📌 Patient demographics and family history contribute to model performance",
        "📌 Metabolic syndrome components show compound predictive effects"
    ]

    for finding in findings:
        print(f"   {finding}")

    print("\n💡 CLINICAL IMPLEMENTATION RECOMMENDATIONS")
    print("=" * 50)

    recommendations = [
        "🔹 Implement baseline HbA1c monitoring as primary screening tool",
        "🔹 Integrate weight management programs with diabetes care protocols",
        "🔹 Establish comprehensive metabolic syndrome risk assessment",
        "🔹 Develop age and gender-specific treatment protocols",
        "🔹 Create family history-based risk stratification systems",
        "🔹 Monitor model performance with real-world clinical data",
        "🔹 Validate predictions with clinical expertise before implementation"
    ]

    for rec in recommendations:
        print(f"   {rec}")

    print("\n⚠️  STUDY LIMITATIONS & FUTURE DIRECTIONS")
    print("=" * 50)

    limitations = [
        "🔸 Model requires external validation on independent patient cohorts",
        "🔸 Temporal validation needed to assess prediction stability over time",
        "🔸 Integration with electronic health records requires further development",
        "🔸 Cost-effectiveness analysis needed for clinical implementation",
        "🔸 Ethical considerations for AI-assisted clinical decision making"
    ]

    for limitation in limitations:
        print(f"   {limitation}")

    future_directions = [
        "🚀 Prospective clinical trial validation",
        "🚀 Real-time prediction system development",
        "🚀 Multi-center validation studies",
        "🚀 Integration with wearable device data",
        "🚀 Development of intervention recommendation systems"
    ]

    print("\n🔮 Future Research Directions:")
    for direction in future_directions:
        print(f"   {direction}")

    print("\n🚀 DEPLOYMENT CONSIDERATIONS")
    print("=" * 40)

    deployment_items = [
        "📋 Model versioning and monitoring systems",
        "🔐 Data privacy and security compliance (HIPAA)",
        "🔄 Continuous model retraining protocols",
        "📊 Performance monitoring dashboards",
        "👥 Clinical staff training and education programs",
        "⚡ Real-time prediction API development",
        "📱 Integration with clinical workflow systems"
    ]

    for item in deployment_items:
        print(f"   {item}")

    print("\n" + "=" * 70)
    print("🎯 CONCLUSION: AutoML analysis provides valuable insights for diabetes management")
    print("💡 NEXT STEPS: Validate findings and prepare for clinical implementation")
    print("📞 CONTACT: Research team available for clinical collaboration")
    print("=" * 70)


# Generate final report
print("🚀 Generating Comprehensive Clinical Research Report...")
print()

try:
    generate_comprehensive_report()
    print("\n📊 The AutoML analysis is now complete.")
    print("🏥 Results are ready for clinical review and validation.")

except Exception as e:
    print(f"❌ Error generating report: {str(e)}")
    print("⚠️  Please ensure all previous cells have been executed")

print("\n" + "="*70)
print("🎉 AUTOML DIABETES HBA1C PREDICTION ANALYSIS COMPLETED")
print("Thank you for using this comprehensive clinical research notebook!")
print("="*70)

In [ ]:
# =============================================================================
# WORKFLOW VALIDATION & READINESS CHECK
# =============================================================================

def validate_notebook_readiness():
    """
    Comprehensive validation of notebook readiness for training
    
    Checks:
    - Data files availability
    - Required columns presence  
    - Memory requirements
    - Feature completeness
    """
    
    print("🔍 NOTEBOOK READINESS VALIDATION")
    print("=" * 50)
    
    validation_results = {
        'data_files': False,
        'target_variable': False, 
        'features_ready': False,
        'memory_ok': False,
        'ready_for_training': False
    }
    
    # 1. Check data files
    print("📂 Checking data file availability...")
    
    dataset_files = NOTEBOOK_CONFIG.get("dataset_files", {})
    missing_files = []
    
    for alias, filename in dataset_files.items():
        filepath = DATA_DIR / filename
        if filepath.exists():
            file_size = filepath.stat().st_size / (1024**2)  # MB
            print(f"   ✅ {filename}: {file_size:.1f} MB")
        else:
            print(f"   ❌ {filename}: Not found")
            missing_files.append(filename)
    
    validation_results['data_files'] = len(missing_files) == 0
    
    # 2. Check target variable
    print(f"\n🎯 Checking target variable...")
    target_col = NOTEBOOK_CONFIG['target_column']
    
    if 'df_main' in globals():
        if target_col in df_main.columns:
            target_completeness = df_main[target_col].notna().mean() * 100
            print(f"   ✅ {target_col}: {target_completeness:.1f}% complete")
            validation_results['target_variable'] = target_completeness >= 80
        else:
            print(f"   ❌ {target_col}: Column not found")
    else:
        print(f"   ⚠️  Data not loaded yet - run data loading cells first")
    
    # 3. Check feature readiness
    print(f"\n📊 Checking feature engineering...")
    
    if 'X' in globals() and 'y' in globals():
        print(f"   ✅ Features prepared: {X.shape[1]} features, {X.shape[0]} samples")
        print(f"   ✅ Target variable ready: {len(y)} samples")
        validation_results['features_ready'] = True
    else:
        print(f"   ⚠️  Feature engineering not completed - run feature engineering cells")
    
    # 4. Check memory requirements  
    print(f"\n💾 Checking memory requirements...")
    
    if 'feature_info' in globals():
        memory_mb = feature_info.get('memory_usage_mb', 0)
        print(f"   📊 Dataset memory: {memory_mb:.1f} MB")
        print(f"   🖥️  Estimated H2O memory needed: {memory_mb * 3:.1f} MB")
        validation_results['memory_ok'] = memory_mb < 1000  # Conservative estimate
    else:
        print(f"   ⚠️  Memory usage not calculated yet")
    
    # 5. Overall readiness assessment
    print(f"\n🎯 OVERALL READINESS ASSESSMENT:")
    print("=" * 30)
    
    checks = [
        ("Data Files Available", validation_results['data_files']),
        ("Target Variable Ready", validation_results['target_variable']), 
        ("Features Engineered", validation_results['features_ready']),
        ("Memory Requirements OK", validation_results['memory_ok'])
    ]
    
    passed_checks = 0
    for check_name, passed in checks:
        status_icon = "✅" if passed else "❌"
        print(f"   {status_icon} {check_name}")
        if passed:
            passed_checks += 1
    
    validation_results['ready_for_training'] = passed_checks >= 3
    
    if validation_results['ready_for_training']:
        print(f"\n🎉 NOTEBOOK IS READY FOR TRAINING!")
        print(f"✅ {passed_checks}/4 validation checks passed")
        print(f"🚀 You can proceed to run the AutoML training cells")
    else:
        print(f"\n⚠️  NOTEBOOK NOT READY FOR TRAINING")  
        print(f"❌ Only {passed_checks}/4 validation checks passed")
        print(f"📋 Please complete the missing requirements above")
    
    return validation_results


# Run validation check
try:
    readiness_check = validate_notebook_readiness()
    
    if readiness_check['ready_for_training']:
        print(f"\n🎯 RECOMMENDED NEXT STEPS:")
        print(f"   1. Run H2O AutoML training (Cell 22)")
        print(f"   2. Analyze feature importance (Cell 23)")  
        print(f"   3. Review final clinical report (Cell 24)")
        print(f"   ⏱️  Total estimated time: 15-20 minutes")
    else:
        print(f"\n🔧 TROUBLESHOOTING STEPS:")
        if not readiness_check['data_files']:
            print(f"   • Check DATA_DIR path: {DATA_DIR}")
            print(f"   • Upload CSV files to final_imputed_data folder")
        if not readiness_check['target_variable']:
            print(f"   • Run data loading cells (Cells 10-16)")
        if not readiness_check['features_ready']:
            print(f"   • Run feature engineering cells (Cells 18-21)")
            
except Exception as e:
    print(f"❌ Validation error: {str(e)}")
    print(f"⚠️  Please run previous cells in sequence before validation")